# EMA kernel micro-benchmark

Compares two implementations of the same EMA recurrence:

- **`builder`** — the current `calc_ema`: `PrimitiveChunkedBuilder` with per-element `append_value` / `append_null`.
- **`collect`** — map over `ca.iter()` into `.collect()`, the approach polars' own `ewm_mean` uses (TrustedLen fast path).

The input has a **leading null**, so the nullable code path (validity bitmap) is exercised — the case where the builder overhead hurts most.

Run top to bottom. The deps cell compiles polars on first run (slow once). `:opt 2` makes this a release-mode comparison.

In [2]:
:opt 2

Optimization: 2


In [3]:
:dep polars = { version = "0.54.4", features = ["dtype-struct"] }
:dep polars-arrow = { version = "0.54.4", default-features = false }

In [4]:
use polars::prelude::*;
use std::time::Instant;

In [5]:
// Build N f64 values (reproducible random walk) with a leading null.
let n = 100_000usize;
let mut data: Vec<Option<f64>> = Vec::with_capacity(n);
data.push(None); // leading null -> forces the nullable path (validity bitmap)
let mut acc = 100.0f64;
let mut state: u64 = 0x9E3779B97F4A7C15;
for _ in 1..n {
    // xorshift64 for reproducible pseudo-random steps
    state ^= state << 13;
    state ^= state >> 7;
    state ^= state << 17;
    let u = (state >> 11) as f64 / ((1u64 << 53) as f64);
    acc += u - 0.5;
    data.push(Some(acc));
}
let ca: Float64Chunked = data.iter().copied().collect();
(ca.len(), ca.null_count())

(100000, 1)

In [6]:
// Current approach: PrimitiveChunkedBuilder, append per element.
fn ema_builder(ca: &Float64Chunked, period: i64) -> Float64Chunked {
    let alpha = 2.0 / (period as f64 + 1.0);
    let mut ema = f64::NAN;
    let mut count: i64 = 0;
    let mut builder = PrimitiveChunkedBuilder::<Float64Type>::new("ema".into(), ca.len());
    for opt_val in ca.iter() {
        let Some(val) = opt_val else {
            ema = f64::NAN;
            count = 0;
            builder.append_null();
            continue;
        };
        if count == 0 { ema = val; } else { ema += alpha * (val - ema); }
        count += 1;
        if count >= period { builder.append_value(ema); } else { builder.append_null(); }
    }
    builder.finish()
}

In [7]:
// Alternative: map into .collect() (TrustedLen), mirroring polars ewm_mean.
fn ema_collect(ca: &Float64Chunked, period: i64) -> Float64Chunked {
    let alpha = 2.0 / (period as f64 + 1.0);
    let mut ema = f64::NAN;
    let mut count: i64 = 0;
    ca.iter()
        .map(|opt_val| match opt_val {
            None => {
                ema = f64::NAN;
                count = 0;
                None
            }
            Some(val) => {
                if count == 0 { ema = val; } else { ema += alpha * (val - ema); }
                count += 1;
                (count >= period).then_some(ema)
            }
        })
        .collect()
}

In [8]:
// Both must produce identical output.
// (explicit types: evcxr persists top-level lets and needs the concrete type)
let a: Float64Chunked = ema_builder(&ca, 20);
let b: Float64Chunked = ema_collect(&ca, 20);
let identical = a.len() == b.len()
    && a.iter().zip(b.iter()).all(|(x, y)| match (x, y) {
        (Some(x), Some(y)) => x == y,
        (None, None) => true,
        _ => false,
    });
println!("identical = {identical}  (len = {}, nulls = {})", a.len(), a.null_count());

identical = true  (len = 100000, nulls = 20)


In [9]:
fn bench<F: FnMut() -> Float64Chunked>(label: &str, runs: usize, mut f: F) {
    std::hint::black_box(f()); // warmup
    let mut best = f64::INFINITY;
    let mut total = 0.0f64;
    for _ in 0..runs {
        let t = Instant::now();
        let r = f();
        std::hint::black_box(&r);
        let us = t.elapsed().as_secs_f64() * 1e6;
        total += us;
        if us < best { best = us; }
    }
    println!("{:<9} min = {:8.1} us   mean = {:8.1} us   ({} runs)",
             label, best, total / runs as f64, runs);
}

In [10]:
let period = 20i64;
bench("builder", 200, || ema_builder(&ca, period));
bench("collect", 200, || ema_collect(&ca, period));

builder   min =    243.3 us   mean =    247.2 us   (200 runs)


collect   min =    424.5 us   mean =    447.7 us   (200 runs)
